# Substructure Training Set Generation

> Notebook purpose: generate multi-substructure synthetic structures, summarize dataset statistics, and visualize samples for QA before model training.

## Workflow
1. Import generation and plotting utilities.
2. Generate dataset and export JSON.
3. Review summary statistics.
4. Visualize random and structured points by substructure.

In [1]:
import json, random, os
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import numpy as np
from typing import List, Dict, Tuple, Optional, Set
import math
rng = random.Random(42)


from Substructure_scripts import build_manual_structure
# Remove the import for _get_corner_removal_indices if present
# from Substructure_scripts import _get_corner_removal_indices



## Generate And Export Multi-Structure Dataset
This section creates the training dataset and saves it as JSON with reproducible settings.

In [ ]:
# Generate multi-structure training set with separate substructures
N_multi = 1000  # Number of multi-structures to generate

rng_multi = random.Random(42)  # Reproducible seed

structures_multi = []
print(f"Generating {N_multi} multi-structures...")

for i in range(N_multi):
    if (i + 1) % 100 == 0:
        print(f"  Generated {i + 1}/{N_multi}")

    # Randomly choose number of substructures (1-3 per multi-structure)
    num_subs = rng_multi.randint(1, 3)

    s = build_manual_structure(
        rng_multi,
        num_substructures=num_subs,
        rows_range=(3, 15),
        cols_range=(3, 15),
        row_gap_domain_mm=(4000, 8000),
        col_gap_domain_mm=(4000, 8000),
        rotation_domain_deg=(-45, 45),
        removal_probability=0.7,
        same_spacing_probability=0.3,
        random_columns_range=(0, 5),  # Add 0-5 random columns per structure
        remove_points_range=(0, 3),   # Remove 0-3 random points per structure
        align_grid_spacing=True,
        align_skew_rotation=True,
        vertical_shift_range=(0, 3),
        structure_type_weights={
            'ortogonal': 0,
            'skewed': 0.2,
            'fan': 0.2,
            'ortogonal_removePart': 0.6,
        },
    )

    structures_multi.append(s)

# Save to JSON
out_file_multi = "JSON/multi_arc_skew_ort_3-15_s3_n1_rn42.json"
os.makedirs("JSON", exist_ok=True)
with open(out_file_multi, "w") as f:
    json.dump({"structures": structures_multi}, f, indent=2)

# Calculate and display statistics
total_points = sum(len(s["points"]) for s in structures_multi)
total_substructures = sum(s["num_substructures"] for s in structures_multi)
avg_points_per_structure = total_points / len(structures_multi)
avg_subs_per_structure = total_substructures / len(structures_multi)

# Count structures with same spacing
same_spacing_count = sum(
    sum(1 for sub in s["substructures"] if sub.get("uses_same_spacing", False))
    for s in structures_multi
)

# Count structure types
structure_types = {}
for s in structures_multi:
    for sub in s["substructures"]:
        stype = sub.get("structure_type", "unknown")
        structure_types[stype] = structure_types.get(stype, 0) + 1

print("\nGeneration complete")
print(f"  Multi-structures generated: {len(structures_multi)}")
print(f"  Total substructures: {total_substructures}")
print(f"  Total points: {total_points}")
print(f"  Avg substructures per multi-structure: {avg_subs_per_structure:.1f}")
print(f"  Avg points per multi-structure: {avg_points_per_structure:.1f}")
print(f"  Avg points per substructure: {total_points/total_substructures:.1f}")

print("\n  Structure type distribution:")
for stype, count in sorted(structure_types.items()):
    print(f"    {stype}: {count} ({count/total_substructures*100:.1f}%)")

print("\n  Shared properties:")
print(f"    Same spacing as main: {same_spacing_count} ({same_spacing_count/total_substructures*100:.1f}%)")

# Show distribution of substructure counts
substructure_counts = [s["num_substructures"] for s in structures_multi]
print("\n  Substructure count distribution:")
for num_subs in sorted(set(substructure_counts)):
    count = substructure_counts.count(num_subs)
    print(f"    {num_subs} substructure(s): {count} ({count/len(structures_multi)*100:.1f}%)")

print(f"\nSaved to: {os.path.abspath(out_file_multi)}")

Generating 10000 multi-structures with separate substructures...
  Generated 100/10000...
  Generated 200/10000...
  Generated 300/10000...
  Generated 400/10000...
  Generated 500/10000...
  Generated 600/10000...
  Generated 700/10000...
  Generated 800/10000...
  Generated 900/10000...
  Generated 1000/10000...
  Generated 1100/10000...
  Generated 1200/10000...
  Generated 1300/10000...
  Generated 1400/10000...
  Generated 1500/10000...
  Generated 1600/10000...
  Generated 1700/10000...
  Generated 1800/10000...
  Generated 1900/10000...
  Generated 2000/10000...
  Generated 2100/10000...
  Generated 2200/10000...
  Generated 2300/10000...
  Generated 2400/10000...
  Generated 2500/10000...
  Generated 2600/10000...
  Generated 2700/10000...
  Generated 2800/10000...
  Generated 2900/10000...
  Generated 3000/10000...
  Generated 3100/10000...
  Generated 3200/10000...
  Generated 3300/10000...
  Generated 3400/10000...
  Generated 3500/10000...
  Generated 3600/10000...
  Genera

## Visualize Generated Structures
Interactive preview of generated structures for quick quality checks.

In [5]:
# Load and visualize the generated multi-structures
in_file_multi = out_file_multi
with open(in_file_multi, "r") as f:
    data_multi = json.load(f)

structures_multi_viz = data_multi["structures"]
print(f"Loaded {len(structures_multi_viz)} multi-structures for visualization")

# Simple plot function for multi-structures
def plot_multi_structure(idx=0):
    s = structures_multi_viz[idx]
    points = s["points"]
    
    # Extract coordinates and substructure IDs
    xs = [p["x_mm"] for p in points]
    ys = [p["y_mm"] for p in points]
    sub_ids = [p["substructure_id"] for p in points]
    
    # Separate grid points from random columns
    grid_points = [(p["x_mm"], p["y_mm"], p["substructure_id"]) for p in points if p["u"] >= 0 and p["v"] >= 0]
    random_points = [(p["x_mm"], p["y_mm"]) for p in points if p["u"] == -1 and p["v"] == -1]
    
    # Create color map for substructures
    unique_subs = sorted(set(sub_ids))
    colors = plt.cm.Set1(np.linspace(0, 1, len(unique_subs)))
    color_map = {sub_id: colors[i] for i, sub_id in enumerate(unique_subs)}
    
    plt.figure(figsize=(12, 10))
    
    # Plot grid points colored by substructure
    if grid_points:
        for sub_id in unique_subs:
            sub_points = [(x, y) for x, y, sid in grid_points if sid == sub_id]
            if sub_points:
                sub_xs, sub_ys = zip(*sub_points)
                plt.scatter(sub_xs, sub_ys, s=100, c=[color_map[sub_id]], 
                           alpha=0.8, edgecolor='black', linewidth=0.5, 
                           label=f'Substructure {sub_id}')
    
    # Plot random columns in red
    if random_points:
        rand_xs, rand_ys = zip(*random_points)
        plt.scatter(rand_xs, rand_ys, s=100, c='red', alpha=0.8, 
                   marker='x', linewidth=2, label='Random columns')
    
    plt.gca().set_aspect("equal")
    
    # Title with info
    num_subs = s.get('num_substructures', len(unique_subs))
    num_grid = len(grid_points)
    num_random = len(random_points)
    
    title = f"Multi-Structure {idx} | {num_subs} substructures, {len(points)} points ({num_grid} grid"
    if num_random > 0:
        title += f" + {num_random} random)"
    else:
        title += ")"
    
    plt.title(title, fontsize=12)
    plt.xlabel("x [mm]")
    plt.ylabel("y [mm]")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

# Create interactive widget
max_display = min(100, len(structures_multi_viz))
print(f"Displaying first {max_display} multi-structures:")

widgets.interact(
    plot_multi_structure,
    idx=widgets.IntSlider(min=0, max=max_display-1, step=1, value=0, 
                         description='Structure:')
)

Loaded 10000 multi-structures for visualization
Displaying first 100 multi-structures:


interactive(children=(IntSlider(value=0, description='Structure:', max=99), Output()), _dom_classes=('widget-i…

<function __main__.plot_multi_structure(idx=0)>